# Pandas Read & Write Data

> 📘 **Python Mastery** · Module 11 — Pandas · Lesson 3/7

Real data lives in files, not in your imagination. This lesson builds a small CSV *by hand* so you never forget what pandas reads, then covers `read_csv` properly — including the parameter that saves you from the infamous mystery `Unnamed: 0` column.

## 🎯 Learning Objectives

- **Create** a working folder with `pathlib` and **build** a CSV file using the plain standard-library `csv` module.
- **Load** CSVs with `pd.read_csv` and inspect what arrived (`head`, `shape`, `dtypes`).
- **Control** loading with `usecols`, `nrows`, `header` / `names`, `index_col`, `dtype` and `na_values`.
- **Explain** why `to_csv(index=False)` matters, and predict the `Unnamed: 0` column when you forget it.
- **Convert** tables to and from JSON with `to_json` / `read_json` using `orient="records"`.
- **Choose** sensibly between CSV, JSON and Excel for storing tabular data.

## 1. Build a Real CSV File First

pandas' readers feel like magic until you remember a CSV is just **text**: one header line, then comma-separated value lines. We will write one ourselves with the standard library's `csv` module — no pandas involved — so the format holds no secrets later.

We keep every file of this course inside a `sample_data/` folder next to the notebook, created with `pathlib`. (`mkdir(exist_ok=True)` means "create unless already there" — safe to re-run.)

**Syntax:**

```python
from pathlib import Path
Path("sample_data").mkdir(exist_ok=True)

with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header_row)     # this row becomes pandas' column names
    writer.writerows(data_rows)
```

In [ ]:
import csv
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)          # project-wide home for our files

rows = [
    ["student_id", "name", "age", "city", "score"],
    ["S001", "Sarah", 22, "Dhaka", 88],
    ["S002", "Rafi", 25, "Sylhet", 92],
    ["S003", "Nabila", 21, "Dhaka", 79],
    ["S004", "Karim", 23, "Chattogram", 95],
]

csv_path = Path("sample_data", "students.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)

print(csv_path.read_text(encoding="utf-8"))       # look inside: plain text!
print("saved ->", csv_path)

## 2. Reading It Back with `pd.read_csv`

One line turns that text file into a full DataFrame: `pd.read_csv(path)`. pandas uses the first line as column names, guesses each column's dtype (numbers become numbers), and attaches a fresh `RangeIndex`.

Always confirm what arrived with `head()` + `shape` + `dtypes` — guessing errors compound downstream.

**Syntax:**

```python
df = pd.read_csv(filepath_or_buffer, sep=",", header=0, ...)
```

In [ ]:
import pandas as pd

df = pd.read_csv("sample_data/students.csv")

print(df.head(3))                 # peek
print()
print("shape:", df.shape)         # 4 rows x 5 columns
print()
print(df.dtypes)                  # age & score arrived as int64 -- good guesses
print()
print(type(df.loc[0, "name"]))    # text columns hold strings

## 3. `read_csv` Parameters Worth Memorizing

`read_csv` has 50+ parameters; these eight cover 95% of real life:

| Parameter | What it does | Example |
|---|---|---|
| `sep` | field delimiter | `sep=";"` for European-style files |
| `header` | which row holds names (`None` if none) | `header=None` |
| `names` | supply your own column names | `names=["name", "score"]` |
| `index_col` | use a column as row labels | `index_col="student_id"` |
| `usecols` | load only some columns | `usecols=["name", "score"]` |
| `nrows` | read only the first n rows | `nrows=100` |
| `dtype` | force dtypes at load time | `dtype={"student_id": str}` |
| `na_values` | extra strings to treat as missing | `na_values=["?", "missing"]` |

The next cells demonstrate the ones that change behavior most.

In [ ]:
import pandas as pd

# Load LESS data: only two columns, only three rows
slim = pd.read_csv(
    "sample_data/students.csv",
    usecols=["name", "score"],   # skip the rest -- less memory, faster
    nrows=3,                     # stop after 3 rows (great for peeking big files)
)
print(slim)

by_id = pd.read_csv("sample_data/students.csv", index_col="student_id")
print()
print(by_id.index.tolist())      # rows are now labeled S001.. instead of 0..3
print(by_id.loc["S002"])         # fetch a student by ID directly

In [ ]:
import csv
from pathlib import Path
import pandas as pd

# A header-less file (raw numbers only) -- common from sensors and legacy systems
raw_path = Path("sample_data", "no_header.csv")
with open(raw_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([["Rafi", 92], ["Mim", 84], ["Tanvir", 77]])

# header=None stops pandas eating the first DATA row as column names;
# names= supplies proper labels.
scores = pd.read_csv(raw_path, header=None, names=["name", "score"])
print(scores)

In [ ]:
import csv
from pathlib import Path
import pandas as pd

# IDs with leading zeros: '0012' is an ID, not the number twelve!
id_path = Path("sample_data", "orders.csv")
with open(id_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["order_id", "amount"],
        ["0012", 450],
        ["0099", 120],
    ])

lazy = pd.read_csv(id_path)
forced = pd.read_csv(id_path, dtype={"order_id": str})   # keep text as text

print(lazy.dtypes)      # order_id guessed as int64 -> zeros LOST
print(forced.dtypes)    # order_id now 'str'        -> '0012' survives
print()
print(forced)

In [ ]:
import csv
from pathlib import Path
import pandas as pd

# Messy values: '?' and 'missing' stand for missing data
messy_path = Path("sample_data", "survey.csv")
with open(messy_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["respondent", "age"],
        ["r1", 29],
        ["r2", "?"],
        ["r3", "missing"],
    ])

default_read = pd.read_csv(messy_path)
cleaner_read = pd.read_csv(messy_path, na_values=["?", "missing"])

print(default_read.dtypes)     # junk strings forced 'age' to become a TEXT column
print(cleaner_read.dtypes)     # with na_values, age is numeric again
print()
print(cleaner_read)
print("missing ages:", cleaner_read["age"].isna().sum())

## 4. Writing CSVs — and the Mystery `Unnamed: 0`

`to_csv(path)` writes any DataFrame back to disk. But by default it also writes the **row labels** as a leading unnamed column. Read that file back and your table grows a ghost column named `Unnamed: 0`.

⚠️ Unless you *want* the index stored, always pass **`index=False`**.

**Syntax:**

```python
df.to_csv(path, index=False)     # data only -- round-trips cleanly
df.to_csv(path)                  # index included -> 'Unnamed: 0' on re-read
```

In [ ]:
import pandas as pd

df = pd.read_csv("sample_data/students.csv")

# The mistake everyone makes once:
df.to_csv("sample_data/students_with_index.csv")          # index silently included
back = pd.read_csv("sample_data/students_with_index.csv")
print(back)
print()
print("mystery column:", back.columns[0], "<- the old row labels came along for the ride")

In [ ]:
import pandas as pd

df = pd.read_csv("sample_data/students.csv")

df.to_csv("sample_data/students_clean.csv", index=False)  # data only
round_trip = pd.read_csv("sample_data/students_clean.csv")

print(round_trip)
print()
print("same shape after round trip:", df.shape == round_trip.shape,
      "| identical columns:", list(df.columns) == list(round_trip.columns))

## 5. JSON: `to_json` / `read_json`

JSON is the language of web APIs. For tabular data the friendliest layout is **`orient="records"`**: a list of objects, one per row (`[{"name": "Sarah", "score": 88}, ...]`). That is exactly the shape most APIs send and accept.

> 🔍 **Under the Hood:** CSV stores *values* but forgets every dtype — everything comes back as text that pandas must re-guess. JSON at least preserves whether something was a number or a string, which is why API pipelines prefer it; neither format remembers your index unless you store it explicitly.

In [ ]:
import json
from pathlib import Path
import pandas as pd

df = pd.read_csv("sample_data/students.csv")
json_path = Path("sample_data", "students.json")

df.to_json(json_path, orient="records", indent=2)
print(json_path.read_text(encoding="utf-8"))       # rows became objects in a list

rebuilt = pd.read_json(json_path, orient="records")
print()
print(rebuilt)
print("round trip ok:", list(rebuilt.columns) == list(df.columns) and rebuilt.shape == df.shape)

## 6. Excel Files (Know It Exists)

Excel workbooks are everywhere in office life, and pandas reads/writes them with `read_excel` / `DataFrame.to_excel` — but that needs the optional `openpyxl` engine, which this course venv does not include, so we do **not** execute these lines.

```bash
pip install openpyxl        # once per environment
```

```python
import pandas as pd

grades = pd.read_excel("sample_data/grades.xlsx", sheet_name="Term 1")   # pick a sheet by name
grades.to_excel("sample_data/summary.xlsx", sheet_name="Summary", index=False)
```

Everything else you learned here (dtypes, `index=False`, inspection) transfers directly.

## 7. Big Files: Read Less, Read Slowly

A 4 GB CSV does not fit in memory and does not need to:

- **Load only what you need** — `usecols` cuts memory proportionally to columns dropped.
- **Stream in chunks** with `chunksize` — each iteration yields a DataFrame of n rows, so you can aggregate piece by piece.
- **Peek first** — read `nrows=5`, check dtypes, then commit to a full load with explicit `dtype=`.

```python
# streaming pattern (illustration -- no big file here):
total = 0
for chunk in pd.read_csv("huge_sales.csv", usecols=["amount"], chunksize=100_000):
    total += chunk["amount"].sum()      # aggregate per chunk, never hold it all
```

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| `df.to_csv(path)` without `index=False` | Re-read shows a ghost `Unnamed: 0` column | Pass `index=False` unless the index carries meaning |
| Trusting dtype guessing | `'0012'` becomes `12`; IDs, phones, postal codes corrupted | Force with `dtype={"col": str}` at load time |
| Ignoring junk values like `"?"` or `"missing"` | Whole numeric column arrives as text | List them in `na_values=[...]` |
| `header=None` forgotten on headerless files | First data row silently becomes the column names | Use `header=None, names=[...]` |
| Relative paths that depend on where you ran Python | `FileNotFoundError` "out of nowhere" | Build paths from `Path(__file__).parent` or run from the notebook's folder |

## 💡 Best Practices & Pro Tips

- After **every** load, run `df.head()` + `df.shape` + `df.dtypes` before doing anything else — thirty seconds that prevent hours of confusion.
- Specify `dtype=` for ID-like columns and `na_values=` for your dataset's placeholder strings *at load time*; fixing types afterwards works but costs more code.
- Always pass `encoding="utf-8"` when writing files yourself, so names like *"Nusrat Jahan"* survive the round trip.
- Keep generated data in a dedicated `sample_data/` folder — notebooks stay reproducible, teammates can find artifacts.
- 🤖 **AI-engineering relevance:** ML datasets are loaded exactly this way; training scripts stream huge CSVs with `chunksize`, and prediction results are commonly logged to `predictions.csv` or `results.json` for later evaluation.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `pd.read_csv(path)` | Text table → DataFrame | `pd.read_csv("students.csv")` |
| `sep`, `header`, `names` | Delimiter & header control | `read_csv(p, sep=";", header=None, names=["a"])` |
| `index_col` | Column becomes row labels | `read_csv(p, index_col="student_id")` |
| `usecols`, `nrows` | Load less data | `read_csv(p, usecols=["name"], nrows=100)` |
| `dtype`, `na_values` | Fix types & missing markers on load | `read_csv(p, dtype={"id": str}, na_values=["?"])` |
| `df.to_csv(path, index=False)` | DataFrame → clean CSV | `df.to_csv("out.csv", index=False)` |
| `to_json` / `read_json` | JSON records ↔ DataFrame | `df.to_json(p, orient="records")` |
| `read_excel` / `to_excel` | Excel workbooks (needs `openpyxl`) | mentioned, not executed |

**Key takeaways**

- A CSV is just labeled text — write one by hand once and readers stop being magic.
- Control loading instead of trusting guesses: `dtype`, `na_values`, `usecols`, `nrows`.
- `index=False` is the default habit for writing; otherwise re-reads sprout `Unnamed: 0`.
- Inspect immediately after any load: `head()`, `shape`, `dtypes`.

## 🔗 Next Lesson

Next up: **[04_Analyzing_Data](../04_Analyzing_Data/notes.ipynb)** — the table is loaded; now interrogate it with describe, value_counts, sorting and correlations.